In [1]:
# Importing packages
import numpy as np
import numpy_financial as npf
import pandas as pd
from prepay_amort import get_cash_flows
from dateutil.relativedelta import relativedelta
from tqdm import tqdm
from datetime import date
from utils import VectorHandler, normalize_next_payment_date

In [ ]:
Base_CPR = "3.66 4.57 5.48 6.39 7.3 8.2 9.11 10.02 11.29 11.87 12.33 12.66 12.87 12.95 12.91 12.74 12.44 12.02 11.48 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5"
Current_CPR = Base_CPR
# CDR = "0 0 0 0 0 2.37 4.7 6.97 8.75 9.08 9.41 9.74 10.06 10.39 10.71 11.04 11.36 11.68 12 12.32 12.64 12.96 13.28 13.59 13.91 14.22 14.53 14.84 15.15 15.46 15.77 16.08 16.38 16.69 16.99 17.3 17.6 17.9 18.2 18.5 18.8 19.09 19.39 19.68 19.98 20.27 20.56 20.85 21.14 21.53 21.53"
# Severity = 70

# BASE (ALWAYS KEEP)
Base_CDR = "0.0000 0.0000 0.0000 0.0000 0.0000 2.2157 4.3897 6.5316 8.2109 8.5219 8.8329 9.1439 9.4557 9.7683 10.0809 10.3943 10.7086 11.0229 11.3381 11.6533 11.9694 12.2856 12.6027 12.9207 13.2388 13.5578 13.8769 14.1969 14.5170 14.8380 15.1591 15.4812 15.8034 16.1265 16.4498 16.7740 17.0984 17.4237 17.7492 18.0756 18.4022 18.7297 19.0574 19.3860 19.7397 19.7397 19.7397 19.7397 19.7397 19.7397"
Base_Severity = "70"

# # BASE
# Current_CDR = Base_CDR
# Current_Severity = Base_Severity

# # BEST
# Current_CDR = "0.0000 0.0000 0.0000 0.0000 0.0000 1.9615 3.8892 5.7882 7.2821 7.5607 7.8401 8.1195 8.3997 8.6807 8.9617 9.2435 9.5253 9.8079 10.0906 10.3741 10.6577 10.9422 11.2267 11.5121 11.7976 12.0840 12.3705 12.6579 12.9454 13.2338 13.5224 13.8110 14.1005 14.3902 14.6808 14.9715 15.2632 15.5550 15.8477 16.1406 16.4345 16.7285 17.0235 17.3186 17.6497 17.6497 17.6497 17.6497 17.6497 17.6497"
# Current_Severity = "65"

# WORST
Current_CDR = "0.0000 0.0000 0.0000 0.0000 0.0000 2.4701 4.8907 7.2757 9.1404 9.4838 9.8272 10.1706 10.5148 10.8590 11.2040 11.5490 11.8948 12.2407 12.5874 12.9342 13.2818 13.6295 13.9780 14.3267 14.6762 15.0258 15.3762 15.7267 16.0780 16.4295 16.7818 17.1342 17.4874 17.8407 18.1949 18.5492 18.9043 19.2596 19.6157 19.9719 20.3289 20.6860 21.0440 21.4020 21.7947 21.7947 21.7947 21.7947 21.7947 21.7947"
Current_Severity = "75"

# COUPON_MULT = "1.000"
COUPON_MULT = "1.000 0.987 0.967 0.956 0.948 0.942 0.937 0.932 0.928 0.925 0.922 0.920 0.917 0.915 0.913 0.911 0.909 0.907 0.906 0.904 0.903 0.901 0.900 0.899 0.898 0.897 0.895 0.894 0.893 0.892 0.891 0.891 0.890 0.889 0.888 0.887 0.886 0.886 0.885 0.884 0.883 0.883 0.882 0.881 0.881 0.880 0.880 0.880 0.880"
TARGET_YEAR = 2024
TARGET_MONTH = 11

# Load
loan_dataset = pd.read_excel('Yields and Prices.xlsx')
# loan_dataset = loan_dataset[loan_dataset['ASTAT'] == 0]
print(max(loan_dataset['ANXDTDT']))

# Convert dates
date_columns = ['ABKDTDT', 'ANXDTDT', 'ACODTDT', 'AUD3DT', 'AMTDTDT']
for col in date_columns:
    loan_dataset[col] = pd.to_datetime(loan_dataset[col])

# Add vectors
loan_dataset['Base_CPR'] = Base_CPR
loan_dataset['Base_CDR'] = Base_CDR
loan_dataset['Base_Severity'] = Base_Severity
loan_dataset['Current_CPR'] = Current_CPR
loan_dataset['Current_CDR'] = Current_CDR
loan_dataset['Current_Severity'] = Current_Severity

loan_dataset['COUPON_MULT'] = COUPON_MULT

# Normalize next payment dates
loan_dataset['ANXDTDT'] = loan_dataset['ANXDTDT'].apply(
    lambda x: normalize_next_payment_date(x, TARGET_YEAR, TARGET_MONTH)
)
print(max(loan_dataset['ANXDTDT']))

loan_dataset.head()

In [ ]:
indx = 0
xddf, _, _ = get_cash_flows(
    apmt1=loan_dataset['APMT1'][indx],
    rempmts=loan_dataset['AOTRM'][indx],
    original_balance=loan_dataset['AOFIN'][indx], 
    unpaid_balance=loan_dataset['AOFIN'][indx], 
    interest_rate=loan_dataset['ARATE'][indx], 
    original_term=loan_dataset['AOTRM'][indx], 
    calculation_start_date=loan_dataset['ACODTDT'][indx],
    next_payment_date=loan_dataset['AUD3DT'][indx], 
    cpr=loan_dataset['Base_CPR'][indx], 
    cdr=loan_dataset['Base_CDR'][indx], 
    severity=loan_dataset['Base_Severity'][indx],
    price=loan_dataset['PRICE'][indx],
    arpay=loan_dataset['ARPAY'][indx],
    xirr_calc=True,
    output=True,
    coupon_mult=loan_dataset['COUPON_MULT'][indx],
    reference_date=loan_dataset['ABKDTDT'][indx]  # Add this line
)
xddf.head()

In [ ]:
xddf.tail()

In [5]:
xddf.to_excel(f'amort_loan{indx}_origination.xlsx')

In [ ]:
i = 0
print(loan_dataset['AOTRM'][i], loan_dataset['REMPMTS'][i], loan_dataset['AOTRM'][i] - loan_dataset['REMPMTS'][i])
xddf, _, _ = get_cash_flows(
            apmt1=loan_dataset['APMT1'][i],
            rempmts=loan_dataset['REMPMTS'][i],
            original_balance=loan_dataset['AOFIN'][i], 
            unpaid_balance=loan_dataset['ANETBAL'][i],
            interest_rate=loan_dataset['ARATE'][i],
            original_term=loan_dataset['AOTRM'][i],
            calculation_start_date=loan_dataset['ABKDTDT'][i],
            next_payment_date=loan_dataset['ANXDTDT'][i],
            cpr=loan_dataset['Current_CPR'][i],
            cdr=loan_dataset['Current_CDR'][i],
            severity=loan_dataset['Current_Severity'][i],
            price=loan_dataset['PRICE'][i],
            output=True,
            arpay=loan_dataset['ARPAY'][i],
            maturity_date=loan_dataset['AMTDTDT'][i],
            contract_date=loan_dataset['ACODTDT'][i],
            coupon_mult=loan_dataset['COUPON_MULT'][i]
        )
xddf[35:44]

In [7]:
xddf.to_excel(f'amort_loan{i}_october.xlsx')

In [8]:
# sum = 0
# while True:
#     sum +=1

In [ ]:
yields = []
priceNa = 0
current_loans = loan_dataset[loan_dataset['ASTAT'] == 0].copy()
for index, row in tqdm(current_loans.iterrows()):
    if pd.notna(row['PRICE']):
        loan_yield = get_cash_flows(
            apmt1=row['APMT1'],
            rempmts=row['AOTRM'],
            original_balance=row['AOFIN'], 
            unpaid_balance=row['AOFIN'], 
            interest_rate=row['ARATE'], 
            original_term=row['AOTRM'], 
            calculation_start_date=row['ACODTDT'],
            next_payment_date=row['AUD3DT'], 
            cpr=row['Base_CPR'], 
            cdr=row['Base_CDR'], 
            severity=row['Base_Severity'],
            price=row['PRICE'],
            arpay=row['ARPAY'],
            xirr_calc=True,
            coupon_mult=row['COUPON_MULT']  # Add this line
        )
    else: 
        priceNa += 1
        loan_yield = None
        
    yields.append(loan_yield)

current_loans.loc[:, 'Original Yields'] = yields

In [ ]:
prices = []
cpr_list = []
cdr_list = []
severity_list = []
coupon_mult_list = []  # Add this line
priceNa = 0
for index, row in tqdm(current_loans.iterrows()):
    if pd.notna(row['PRICE']):
        npv, cpr, cdr, severity, coupon_mult = get_cash_flows(  # Update unpacking
            apmt1=row['APMT1'],
            rempmts=row['REMPMTS'],
            original_balance=row['AOFIN'],
            unpaid_balance=row['ANETBAL'],
            interest_rate=row['ARATE'],
            original_term=row['AOTRM'],
            calculation_start_date=row['ABKDTDT'],
            next_payment_date=row['ANXDTDT'],
            cpr=row['Current_CPR'],
            cdr=row['Current_CDR'],
            severity=row['Current_Severity'],
            price=row['PRICE'],
            original_yield=row['Original Yields'],
            arpay=row['ARPAY'],
            contract_date=row['ACODTDT'],
            maturity_date=row['AMTDTDT'],
            coupon_mult=row['COUPON_MULT'],
            reference_date=row['ABKDTDT'] # Add this line
        )
        original_balance = row['ANETBAL'] if row['ANETBAL'] > 0 else 0
        price = npv*100/row['ANETBAL']
    else:
        priceNa += 1
        price = None
        
    prices.append(price)
    cpr_list.append(cpr)
    cdr_list.append(cdr)
    severity_list.append(severity)
    coupon_mult_list.append(coupon_mult)  # Add this line

current_loans.loc[:, 'Trimmed CPR'] = cpr_list
current_loans.loc[:, 'Trimmed CDR'] = cdr_list
current_loans.loc[:, 'Trimmed Severity'] = severity_list
current_loans.loc[:, 'Trimmed COUPON_MULT'] = coupon_mult_list
current_loans.loc[:, 'New Price'] = prices

current_loans.to_excel('Original Yields and New Prices.xlsx', index=False)


In [ ]:
from pandas import Timestamp

# Initialize lists for each monthboard
oct_dfs07 = []
orig_dfs07 = []
oct_dfs08 = []
orig_dfs08 = []
oct_dfs09 = []
orig_dfs09 = []
oct_dfs10 = []
orig_dfs10 = []
oct_dfs11 = []
orig_dfs11 = []
oct_dfs12 = []
orig_dfs12 = []

for _, row in tqdm(loan_dataset.iterrows()):
    if pd.notna(row['PRICE']):
        is_current = (row["ASTAT"] == 0)
        if is_current:
            oct31_df, _, _ = get_cash_flows(
                apmt1=row['APMT1'],
                rempmts=row['REMPMTS'],
                original_balance=row['AOFIN'], 
                unpaid_balance=row['ANETBAL'],
                interest_rate=row['ARATE'],
                original_term=row['AOTRM'],
                calculation_start_date=row['ABKDTDT'],
                next_payment_date=row['ANXDTDT'],
                cpr=row['Current_CPR'],
                cdr=row['Current_CDR'],
                severity=row['Current_Severity'],
                price=row['PRICE'],
                maturity_date=row['AMTDTDT'],
                contract_date=row['ACODTDT'],
                arpay=row['ARPAY'],
                output=True,
                coupon_mult=row['COUPON_MULT'],
                reference_date=row['ABKDTDT']  # Add this line
            )

        start_date = row['ACODTDT']
        next_payment_date = row['AUD3DT']

        # Add conditions for new monthboards
        if row['MONTHBOARD'] == 202307:
            if start_date < Timestamp('2023-07-01'):
                start_date = Timestamp('2023-07-01')
                if next_payment_date < Timestamp('2023-07-01'):
                    next_payment_date = Timestamp('2023-07-02')
        if row['MONTHBOARD'] == 202308:
            if start_date < Timestamp('2023-08-01'):
                start_date = Timestamp('2023-08-01')
                if next_payment_date < Timestamp('2023-08-01'):
                    next_payment_date = Timestamp('2023-08-02')
        if row['MONTHBOARD'] == 202309:
            if start_date < Timestamp('2023-09-01'):
                start_date = Timestamp('2023-09-01')
                if next_payment_date < Timestamp('2023-09-01'):
                    next_payment_date = Timestamp('2023-09-02')
        if row['MONTHBOARD'] == 202310:
            if start_date < Timestamp('2023-10-01'):
                start_date = Timestamp('2023-10-01')
                if next_payment_date < Timestamp('2023-10-01'):
                    next_payment_date = Timestamp('2023-10-02')
        if row['MONTHBOARD'] == 202311:
            if start_date < Timestamp('2023-11-01'):
                start_date = Timestamp('2023-11-01')
                if next_payment_date < Timestamp('2023-11-01'):
                    next_payment_date = Timestamp('2023-11-02')
        if row['MONTHBOARD'] == 202312:
            if start_date < Timestamp('2023-12-01'):
                start_date = Timestamp('2023-12-01')
                if next_payment_date < Timestamp('2023-12-01'):
                    next_payment_date = Timestamp('2023-12-02')

        origination_df, _, _ = get_cash_flows(
            apmt1=row['APMT1'],
            rempmts=row['AOTRM'],
            original_balance=row['AOFIN'],
            unpaid_balance=row['AOFIN'],
            interest_rate=row['ARATE'],
            original_term=row['AOTRM'],
            calculation_start_date=start_date,
            next_payment_date=next_payment_date, 
            cpr=row['Base_CPR'],
            cdr=row['Base_CDR'], 
            severity=row['Base_Severity'],
            price=row['PRICE'],
            output=True,
            arpay=row['ARPAY'],
            coupon_mult=row['COUPON_MULT'],
            reference_date=row['ABKDTDT']  # Add this line
        )

        # Add conditions for new monthboards
        if row['MONTHBOARD'] == 202307:
            if is_current:
                oct_dfs07.append(oct31_df)
            orig_dfs07.append(origination_df)
        if row['MONTHBOARD'] == 202308:
            if is_current:
                oct_dfs08.append(oct31_df)
            orig_dfs08.append(origination_df)
        if row['MONTHBOARD'] == 202309:
            if is_current:
                oct_dfs09.append(oct31_df)
            orig_dfs09.append(origination_df)
        if row['MONTHBOARD'] == 202310:
            if is_current:
                oct_dfs10.append(oct31_df)
            orig_dfs10.append(origination_df)
        if row['MONTHBOARD'] == 202311:
            if is_current:
                oct_dfs11.append(oct31_df)
            orig_dfs11.append(origination_df)
        if row['MONTHBOARD'] == 202312:
            if is_current:
                oct_dfs12.append(oct31_df)
            orig_dfs12.append(origination_df)
    else:
        continue

print("Converting to Monthly")

def convert_to_monthly(df):
    # Convert the Date column to datetime
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Group by year and month, and then apply custom aggregations
    aggregations = {
        'UPB': 'max',
        'Performing UPB': 'max',
        'Gross Charge Offs': 'sum',
        'Net Loss': 'sum',
        'Recoveries': 'sum',
        'Scheduled Principal': 'sum',
        'Voluntary Prepayments': 'sum',
        'Principal Cash Flow': 'sum',
        'Interest Cash Flow': 'sum',
        'Total Cash Flow': 'sum'
    }
    
    # Group and aggregate
    df_grouped = df.groupby(df['Date'].dt.to_period("M")).agg(aggregations).reset_index()
    
    # Ensure chronological order
    df_grouped.sort_values('Date', inplace=True)
    
    # Convert the Date from Period to datetime (start of the period)
    df_grouped['Date'] = df_grouped['Date'].dt.to_timestamp()

    # Check for and insert missing months
    all_months = pd.date_range(start=df_grouped['Date'].min(), end=df_grouped['Date'].max(), freq='MS')
    missing_months = all_months.difference(df_grouped['Date'])
    
    # For each missing month, insert a row with values from the first row
    for missing_month in missing_months:
        # Find the previous month to the missing month in the dataset
        prev_month = missing_month - pd.DateOffset(months=1)
        if prev_month in df_grouped['Date'].values:
            prev_month_row = df_grouped.loc[df_grouped['Date'] == prev_month].copy()
            prev_month_row['Date'] = missing_month
            df_grouped = pd.concat([df_grouped, prev_month_row], ignore_index=True)
        else:
            # If the missing month is the first month, duplicate the first row with the new month
            first_row = df_grouped.iloc[0].copy()
            first_row['Date'] = missing_month
            df_grouped = pd.concat([df_grouped, pd.DataFrame([first_row])], ignore_index=True)

    # Sort again after inserting rows
    df_grouped.sort_values('Date', inplace=True)
    df_grouped.reset_index(drop=True, inplace=True)

    return df_grouped

def combine_dfs_by_month(dfs):
    for df in dfs:
        df['Date'] = pd.to_datetime(df['Date'])
    for df in dfs:
        df.set_index('Date', inplace=True)

    combined_df = pd.concat(dfs)

    monthly_sum = combined_df.resample('ME').sum()
    monthly_sum.reset_index(inplace=True)

    return monthly_sum

# Update lists to include new monthboards
dfs = [oct_dfs07, orig_dfs07, oct_dfs08, orig_dfs08, oct_dfs09, orig_dfs09,
       oct_dfs10, orig_dfs10, oct_dfs11, orig_dfs11, oct_dfs12, orig_dfs12]
names = ['oct_dfs07', 'orig_dfs07', 'oct_dfs08', 'orig_dfs08', 'oct_dfs09', 'orig_dfs09',
         'oct_dfs10', 'orig_dfs10', 'oct_dfs11', 'orig_dfs11', 'oct_dfs12', 'orig_dfs12']

combined_dfs = []
for df in tqdm(dfs):
    new_dfs = []
    for sub_df in df:
        new_dfs.append(convert_to_monthly(sub_df))
    combined_dfs.append(combine_dfs_by_month(new_dfs))

# Save to Excel
print("Saving to Excel")
for i, df in enumerate(dfs):
    combined_dfs[i].to_excel(f'monthboard_{names[i]}.xlsx', index=False)